In [1]:
import os
import re
import requests
import pandas as pd
import logging
from dotenv import load_dotenv
from datetime import datetime, timedelta
from time import sleep, time as now

# Logging setup
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# Output directory and file paths
output_dir = r"C:\Android Mobile App\Step1_URL_Search"
output_csv_all = os.path.join(output_dir, "github_android_search_results_all_Fork.csv")
output_csv_filtered = os.path.join(output_dir, "github_android_search_results_filtered_Fork.csv")
os.makedirs(output_dir, exist_ok=True)

# Load GitHub tokens
load_dotenv("All_Tokens.env")
tokens = [v for k, v in os.environ.items() if k.startswith("GITHUB_TOKEN_") and v]
if not tokens:
    raise ValueError("No GitHub tokens found in All_Tokens.env")

token_index = 0
HEADERS = {
    "Authorization": f"token {tokens[token_index]}",
    "Accept": "application/vnd.github.mercy-preview+json",
    "User-Agent": "android-repo-crawler/1.0"
}

# Define date range for search
start_date = datetime.strptime("2010-01-01", "%Y-%m-%d")
end_date = datetime.strptime("2010-01-31", "%Y-%m-%d")

# Window settings to adjust based on search result volume
initial_window_hours = 15 * 24
min_window_hours = 1
max_window_hours = 90 * 24
MAX_RESULTS_PER_QUERY = 1000
TARGET_FILL_RATIO = 0.25

# GitHub search queries
base_queries = [
    "stars:>50 language:Kotlin fork:true archived:false",
    "stars:>50 language:Java fork:true archived:false",
    "stars:>50 language:Dart fork:true archived:false",
    "stars:>50 topic:android fork:true archived:false",
    "stars:>50 android in:name,description,readme fork:true archived:false",
]

# Rotate token if API rate limit is hit
def rotate_token():
    global token_index, HEADERS
    token_index = (token_index + 1) % len(tokens)
    HEADERS = {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github.mercy-preview+json",
    }
    logger.warning(f"Rotated → Token #{token_index + 1} / {len(tokens)}")

# Check GitHub API rate limits
def check_rate_limit():
    r = requests.get("https://api.github.com/rate_limit", headers=HEADERS)
    if r.status_code != 200:
        logger.warning("Could not check rate limit.")
        return
    data = r.json()
    remaining = data['resources']['search']['remaining']
    reset_epoch = data['resources']['search']['reset']
    reset_in = max(0, reset_epoch - now())
    logger.info(f"Remaining: {remaining} | Resets in {reset_in/60:.1f} min")
    if remaining < 5:
        if len(tokens) > 1:
            rotate_token()
            check_rate_limit()
        else:
            logger.warning(f"No extra tokens. Sleeping {reset_in/60:.1f} min")
            sleep(reset_in + 5)

# Estimate the number of search results for a query
def check_count(query):
    while True:
        check_rate_limit()
        r = requests.get("https://api.github.com/search/repositories",
                         headers=HEADERS, params={"q": query, "per_page": 1})
        if r.status_code == 200:
            return r.json().get("total_count", 0)
        elif r.status_code == 403:
            rotate_token()
        else:
            logger.error(f"Count error: {r.status_code} — {r.text}")
            return -1

# Fetch repository metadata for a given query
def fetch_items(query):
    all_items = []
    for page in range(1, 11):
        check_rate_limit()
        r = requests.get("https://api.github.com/search/repositories",
                         headers=HEADERS, params={"q": query, "per_page": 100, "page": page})
        if r.status_code == 200:
            items = r.json().get("items", [])
            if not items:
                break
            all_items.extend(items)
            sleep(1)
        elif r.status_code == 403:
            rotate_token()
            return fetch_items(query)
        else:
            logger.error(f"Fetch error: {r.status_code} — {r.text}")
            break
    return all_items

# Main execution
final_results = []
for base_query_prefix in base_queries:
    current_start = start_date
    window_hours = initial_window_hours

    while current_start < end_date:
        current_end = min(current_start + timedelta(hours=window_hours), end_date)
        date_range = f"created:{current_start.isoformat()}..{current_end.isoformat()}"
        base_query = f"{base_query_prefix} {date_range}"

        total_count = check_count(base_query)
        logger.info(f"{base_query} → {total_count} repos")

        if total_count >= MAX_RESULTS_PER_QUERY and window_hours > min_window_hours:
            window_hours = max(window_hours // 2, min_window_hours)
            continue
        elif total_count < MAX_RESULTS_PER_QUERY * TARGET_FILL_RATIO and window_hours * 2 <= max_window_hours:
            window_hours = min(window_hours * 2, max_window_hours)

        items = fetch_items(base_query)
        expected_fork = 'fork:true' in base_query_prefix
        expected_archived = 'archived:true' in base_query_prefix
        expected_language = ''
        expected_topic = ''
        expected_keyword = ''

        if 'language:Kotlin' in base_query_prefix:
            expected_language = ['kotlin', 'java', 'dart']
        elif 'language:Java' in base_query_prefix:
            expected_language = ['kotlin', 'java', 'dart']
        elif 'language:Dart' in base_query_prefix:
            expected_language = ['kotlin', 'java', 'dart']
        elif 'topic:android' in base_query_prefix:
            expected_topic = 'android'
        elif 'android' in base_query_prefix:
            expected_keyword = 'android'

        count_passed = 0
        for item in items:
            item["base_qualifier"] = base_query_prefix
            clean_item = {
                "name": item.get("name"),
                "full_name": item.get("full_name"),
                "language": item.get("language"),
                "stargazers_count": item.get("stargazers_count"),
                "forks": item.get("forks"),
                "topics": item.get("topics"),
                "fork": item.get("fork"),
                "archived": item.get("archived"),
                "owner.login": item.get("owner", {}).get("login"),
                "html_url": item.get("html_url"),
                "clone_url": item.get("clone_url"),
                "visibility": item.get("visibility"),
                "size": item.get("size"),
                "open_issues_count": item.get("open_issues_count"),
                "base_qualifier": base_query_prefix,
                "search_qualifier": base_query,
                "repo_stars": item.get("stargazers_count", 0)
            }
            final_results.append(clean_item)
            count_passed += 1

        logger.info(f"Window done: {count_passed} items")
        current_start = current_end + timedelta(seconds=1)

# Save results to CSV and Excel
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
df_clean = pd.DataFrame(final_results)

print("Columns:", list(df_clean.columns))
print(df_clean.head(5))

output_csv = os.path.join(output_dir, f"search_results_{ts}.csv")
output_xlsx = os.path.join(output_dir, f"search_results_{ts}.xlsx")

df_clean.to_csv(output_csv, index=False)
df_clean.to_excel(output_xlsx, index=False)

logger.info(f"All done! Clean results saved to: {output_csv} and {output_xlsx}")
print(f"Final rows saved: {len(df_clean)}")


[INFO] Remaining: 30 | Resets in 1.0 min
[INFO] stars:>50 language:Kotlin fork:true archived:false created:2010-01-01T00:00:00..2010-01-16T00:00:00 → 0 repos
[INFO] Remaining: 29 | Resets in 1.0 min
[INFO] Window done: 0 items
[INFO] Remaining: 28 | Resets in 1.0 min
[INFO] stars:>50 language:Kotlin fork:true archived:false created:2010-01-16T00:00:01..2010-01-31T00:00:00 → 1 repos
[INFO] Remaining: 27 | Resets in 1.0 min
[INFO] Remaining: 26 | Resets in 1.0 min
[INFO] Window done: 1 items
[INFO] Remaining: 25 | Resets in 1.0 min
[INFO] stars:>50 language:Java fork:true archived:false created:2010-01-01T00:00:00..2010-01-16T00:00:00 → 13 repos
[INFO] Remaining: 24 | Resets in 1.0 min
[INFO] Remaining: 23 | Resets in 0.9 min
[INFO] Window done: 13 items
[INFO] Remaining: 22 | Resets in 0.9 min
[INFO] stars:>50 language:Java fork:true archived:false created:2010-01-16T00:00:01..2010-01-31T00:00:00 → 9 repos
[INFO] Remaining: 21 | Resets in 0.9 min
[INFO] Remaining: 20 | Resets in 0.9 min

Columns: ['name', 'full_name', 'language', 'stargazers_count', 'forks', 'topics', 'fork', 'archived', 'owner.login', 'html_url', 'clone_url', 'visibility', 'size', 'open_issues_count', 'base_qualifier', 'search_qualifier', 'repo_stars']
                      name                         full_name language  \
0            quran_android               quran/quran_android   Kotlin   
1          sms-backup-plus           jberkel/sms-backup-plus     Java   
2  android-mapviewballoons  jgilfelt/android-mapviewballoons     Java   
3                 TeleHash                quartzjer/TeleHash     Java   
4                      JPC                    ianopolous/JPC     Java   

   stargazers_count  forks                                             topics  \
0              2173    918                                   [android, quran]   
1              1847    497  [android, android-backup, automatic-backups, c...   
2               850    349                                                 []   
